In [1]:
import sys, os
import numpy as np
from typing import List, Optional

assignment_root = os.path.abspath(os.getcwd())
if assignment_root not in sys.path:
    sys.path.insert(0, assignment_root)
print("Added to sys.path:", assignment_root)

from fixedincomelib import *
print("Fixed Income Library is loaded.")

Added to sys.path: /Users/syonamehra/Desktop/FRE-GY-9743-Assignment-1
Fixed Income Library is loaded.


## Homework 1 --- 1-D Interpolation

Implement the four methods marked `## TODO` inside `Interpolator1DPCP`, in
`fixedincomelib/utilities/numerics.py`:

- `interpolate`
- `integrate`
- `gradient_wrt_ordinate`
- `gradient_of_integrated_value_wrt_ordinate`

Then fill in `bump_reval_interpolator_integrand` further down in this notebook.

The interpolation convention is spelled out in the `Interpolator1DPCP`
docstring. Read it before writing.

To check yourself, run every cell in this notebook top to bottom. Each check
prints your value next to the expected one --- every `diff` should be around `0.0`.


### Test interpolation

In [2]:
axis1 = [1, 3, 5, 7]
values = [3, 4, 5, 6]
interp_method = 'PIECEWISE_CONSTANT_LEFT_CONTINUOUS'
extrap_method = 'FLAT'
interp_1d = qfCreate1DInterpolator(axis1, values, interp_method, extrap_method)

test_points = [
    (0.5, 3.0),   # left wing, flat extrapolation
    (1.0, 3.0),   # exactly on the first node
    (1.5, 4.0),   # inside (1, 3]
    (3.0, 4.0),   # exactly on an interior node
    (5.5, 6.0),   # inside (5, 7]
    (6.5, 6.0),   # inside (5, 7]
    (8.0, 6.0),   # right wing, flat extrapolation
]

for x, expected in test_points:
    v = qfInterpolate1D(x, interp_1d)
    print(f'f({x}) = {v}, expected {expected}, diff = {v - expected}')

f(0.5) = 3, expected 3.0, diff = 0.0
f(1.0) = 3, expected 3.0, diff = 0.0
f(1.5) = 4, expected 4.0, diff = 0.0
f(3.0) = 4, expected 4.0, diff = 0.0
f(5.5) = 6, expected 6.0, diff = 0.0
f(6.5) = 6, expected 6.0, diff = 0.0
f(8.0) = 6, expected 6.0, diff = 0.0


### Test integration of the interpolation

Both endpoints may land anywhere: inside a bucket, on a node, or out in
either flat wing.

In [3]:
integration_cases = [
    ((0.5, 0.9),   1.2),   # both inside the left wing
    ((0.5, 1.2),   2.3),   # left wing into the first bucket
    ((0.5, 3.2),  10.5),   # left wing across into the middle
    ((1.5, 5.2),  17.2),   # entirely inside the node range
    ((3.5, 7.2),  20.7),   # middle bucket out into the right wing
    ((6.0, 7.2),   7.2),   # last bucket into the right wing
    ((8.0, 10.0), 12.0),   # both inside the right wing
    ((0.1, 10.0), 50.7),   # spanning everything
]

for (x_s, x_e), expected in integration_cases:
    v = qfInterpolate1DIntegral(x_s, x_e, interp_1d)
    print(f'integral over [{x_s}, {x_e}] = {v}, expected {expected}, diff = {v - expected}')

integral over [0.5, 0.9] = 1.2000000000000002, expected 1.2, diff = 2.220446049250313e-16
integral over [0.5, 1.2] = 2.3, expected 2.3, diff = 0.0
integral over [0.5, 3.2] = 10.5, expected 10.5, diff = 0.0
integral over [1.5, 5.2] = 17.200000000000003, expected 17.2, diff = 3.552713678800501e-15
integral over [3.5, 7.2] = 20.700000000000003, expected 20.7, diff = 3.552713678800501e-15
integral over [6.0, 7.2] = 7.200000000000001, expected 7.2, diff = 8.881784197001252e-16
integral over [8.0, 10.0] = 12.0, expected 12.0, diff = 0.0
integral over [0.1, 10.0] = 50.7, expected 50.7, diff = 0.0


## Sensitivities

Implement the two analytic sensitivity methods so that they agree with a
bump-and-reval reference:

- `gradient_wrt_ordinate`
- `gradient_of_integrated_value_wrt_ordinate`

`bump_reval_interpolator` below is a worked bump-and-reval for the interpolated
value. Mirror its structure to fill in `bump_reval_interpolator_integrand` for
the integral, then contrast both against your analytic results.

In [4]:
def bump_reval_interpolator(
    x : float,
    axis1 : List,
    values : List,
    interp_method : str,
    extrap_method : str,
    bump_size : Optional[float] = 1e-4):

    base_interpolator = qfCreate1DInterpolator(axis1, values, interp_method, extrap_method)
    b_value = qfInterpolate1D(x, base_interpolator)

    grad = []
    for i in range(len(values)):
        values[i] += bump_size
        this_interp = qfCreate1DInterpolator(axis1, values, interp_method, extrap_method)
        bumped_value = qfInterpolate1D(x, this_interp)
        grad.append((bumped_value - b_value) / bump_size)
        values[i] -= bump_size

    return np.array(grad)


def bump_reval_interpolator_integrand(
    x_s : float,
    x_e : float,
    axis1 : List,
    values : List,
    interp_method : str,
    extrap_method : str,
    bump_size : Optional[float] = 1e-4):

    base_interpolator = qfCreate1DInterpolator(axis1, values, interp_method, extrap_method)
    b_value = qfInterpolate1DIntegral(x_s, x_e, base_interpolator)

    grad = []
    for i in range(len(values)):
        values[i] += bump_size
        this_interp = qfCreate1DInterpolator(axis1, values, interp_method, extrap_method)
        bumped_value = qfInterpolate1DIntegral(x_s, x_e, this_interp)
        grad.append((bumped_value - b_value) / bump_size)
        values[i] -= bump_size

    return np.array(grad)

### Interpolation sensitivity

In [5]:
for x, _ in test_points:
    grad_analytic = qfInterpolate1DGrad(x, interp_1d)
    grad_br = bump_reval_interpolator(x, axis1, values, interp_method, extrap_method)
    print(f'x = {x}: max abs diff = {np.max(np.abs(grad_analytic - grad_br))}')

x = 0.5: max abs diff = 2.1103119252074976e-12
x = 1.0: max abs diff = 2.1103119252074976e-12
x = 1.5: max abs diff = 2.1103119252074976e-12
x = 3.0: max abs diff = 2.1103119252074976e-12
x = 5.5: max abs diff = 2.3305801732931286e-12
x = 6.5: max abs diff = 2.3305801732931286e-12
x = 8.0: max abs diff = 2.3305801732931286e-12


### Integrated interpolation sensitivity

In [6]:
for (x_s, x_e), _ in integration_cases:
    grad_analytic = qfInterpolate1DIntegralGrad(x_s, x_e, interp_1d)
    grad_br = bump_reval_interpolator_integrand(
        x_s, x_e, axis1, values, interp_method, extrap_method)
    print(f'[{x_s}, {x_e}]: max abs diff = {np.max(np.abs(grad_analytic - grad_br))}')

[0.5, 0.9]: max abs diff = 4.000133557724439e-13
[0.5, 1.2]: max abs diff = 1.3102852136626097e-12
[0.5, 3.2]: max abs diff = 7.571721027943568e-12
[1.5, 5.2]: max abs diff = 2.7955415760061442e-11
[3.5, 7.2]: max abs diff = 2.1259438653942198e-11
[6.0, 7.2]: max abs diff = 1.0205170042354439e-12
[8.0, 10.0]: max abs diff = 4.661160346586257e-12
[0.1, 10.0]: max abs diff = 1.1823431123048067e-10


# Bond Forward Business: Written Answers

Notation follows the assignment sheet: trade date `0`, settlement date `Ts` (1y),
underlying bond maturity `Tm`, forward strike `K`, spot bond price `B(0;0,Tm)`,
forward bond price `B(0;Ts,Tm)`, term repo rate `R` for the period `[0,Ts]`
with year fraction `τ = Ts`.

## Q1: Client motivation, term sheet, settlement mechanics

**Motivation.** A client buying a 1,000,000 notional 1-year forward on the
UST 10-year is almost always managing a *future* interest-rate exposure
without moving cash or balance sheet today. Typical cases:

- **Locking in today's yield/price** for money the client will receive in a
  year (e.g., a pension fund expecting a contribution, an insurer expecting
  premium income) so reinvestment risk is hedged now instead of at delivery.
- **Hedging duration on an existing book**: adding or shedding rate
  sensitivity synthetically instead of transacting in the cash bond, which
  avoids market impact, avoids realizing gains/losses for accounting or tax
  purposes, and doesn't tie up cash today.
- **Directional rate view**: a forward requires no upfront principal (only
  margin/credit support under the CSA), so it is a capital-efficient way to
  express a view on 10-year yields a year out.
- **Relative value / basis trades** against the forward curve, futures, or
  other maturities.

**Term sheet parameters.** Notional face amount ($1,000,000); the specific
deliverable bond (CUSIP of the on-the-run/off-the-run UST 10-year, its coupon
and maturity `Tm`); trade date and settlement date `Ts`; the forward strike
`K` (quoted as a price or yield); settlement type (physical delivery vs. cash
settlement); day-count and quotation conventions; the governing ISDA and CSA
(collateral posting, initial/variation margin, discounting rate implied by
the CSA); and the counterparties.

**At settlement `Ts`.** Under physical settlement: Bank A delivers
$1,000,000 face of the UST 10-year to the client, and the client pays Bank A
the agreed amount `K`. That is exactly the payoff in equation (1):
`V(Ts) = B(Ts;Ts,Tm) − K`. The client ends up long a bond worth
`B(Ts;Ts,Tm)` having paid only `K` for it.

## Q2: Why hedge with the spot bond, and how it's funded internally

Bank A is short the forward the instant it agrees to sell the client the bond
at `K` in a year: if yields fall (bond price rises) before `Ts`, the desk
would have to source a more expensive bond to deliver at the old strike.
Buying the actual cash bond today at `B(0;0,Tm)` is the only *exact* hedge,
not a proxy like a future or a different maturity, because it is the same
CUSIP the desk is contractually obligated to deliver, so the price risk
between `0` and `Ts` cancels exactly (this is a static replication argument,
not a model-dependent hedge).

**Internal funding.** The trading desk doesn't have $B(0;0,Tm) million of
free cash lying around, so it borrows it from the internal treasury desk, and
simultaneously term-repos the bond out to the internal repo desk: it posts
the newly bought bond as collateral and receives cash back for the term
`[0,Ts]`, which is used to repay/offset the treasury funding. The desk keeps
economic ownership of the bond (and therefore its coupons, passed through as
"manufactured payments" under the repo) while only paying repo interest as
its net financing cost.

## Q3: Forward price from spot price, coupon schedule, and repo rate

Borrowing $1 for the term `[0,Ts]` at simple repo rate `R` means repaying
`(1 + Rτ)` at `Ts`. So financing `B(0;0,Tm)` dollars of bond purchase this way
means owing `B(0;0,Tm)·(1+Rτ)` at `Ts`.

Any coupon `c_i` paid by the bond at time `t_i ∈ (0,Ts]` is passed through to
the desk (it is still the economic owner under repo), and can be reinvested
at the repo rate for the remaining time to `Ts`, so it grows to
`c_i·(1+R(Ts−t_i))` by `Ts`. Since coupon income reduces the net amount the
desk needs to finance, it is subtracted:

```
B(0;Ts,Tm) = B(0;0,Tm)·(1 + R·τ) − Σᵢ cᵢ·(1 + R·(Ts − tᵢ))
```

This is the standard cost-of-carry identity: **forward = spot + financing
cost − carry income**. Equivalently, discounting instead of compounding:

```
B(0;Ts,Tm) = [ B(0;0,Tm) − Σᵢ cᵢ / (1 + R·tᵢ) ] · (1 + R·τ)
```

## Q4: MTM at any `t ∈ (0,Ts]`

Recompute the forward price at `t` the same way as in Q3, but anchored at
`t`: use the then-current spot price `B(t;t,Tm)`, the then-current repo rate
for the *remaining* term `[t,Ts]`, and only the coupons still to be paid in
`(t,Ts]`. Call this `B(t;Ts,Tm)`. The MTM is then

```
V(t) = df^csa(t,Ts) · ( B(t;Ts,Tm) − K )
```

i.e. the updated forward value minus the locked-in strike, discounted from
`Ts` back to `t` on the CSA curve (the curve implied by whatever collateral
the trade's credit support annex specifies, typically SOFR/OIS).

## Q5: Market risk and where the revenue comes from

Because the desk immediately buys the exact deliverable bond and finances it
in repo, its net position is flat to the bond's price/yield: it has no
material directional market risk. That also means it earns no directional
P&L from the hedge itself. Revenue instead comes from:

- **The bid/offer built into `K`**: the desk quotes the client a strike
  slightly worse than its own fair-value forward (computed via Q3), pocketing
  the spread.
- **Repo specialness.** Benchmark on-the-run Treasuries frequently trade
  "special" in repo (i.e., desks can borrow cash against them at a rate
  *below* general collateral, because everyone wants to borrow that specific
  bond). If the desk quotes the client's forward off a general-collateral
  repo assumption but can actually finance the position on special, it earns
  the GC-minus-special spread over the life of the trade.
- **Residual basis/roll risk** if financing is actually done via a sequence
  of overnight repos rather than one locked term repo: a small, manageable
  risk rather than the bond's outright duration risk.

So the honest pitch to the client is: "you are paying for certainty and
capital efficiency, not for us taking a view; our price reflects our real
funding cost plus a service spread."

## *Q5: Connection to risk-neutral pricing

The Q3 cost-of-carry formula is a replication (static hedge) argument, but
it is exactly the risk-neutral forward price under a change of numeraire.
Using the `Ts`-maturity zero-coupon bond as numeraire, no-arbitrage forces

```
B(0;Ts,Tm) = [ B(0;0,Tm) − PV(coupons) ] / P(0,Ts)
```

where `P(0,Ts) = 1/(1+Rτ)` is the discount factor implied by the repo rate.
This is algebraically identical to the Q3 formula. The reason: constructing
the replicating portfolio (buy the bond, fund it in repo, collect and
reinvest coupons) is precisely what the change-of-numeraire/risk-neutral
argument proves must hold for the market to be arbitrage-free: they are two
derivations of the same no-arbitrage price, not two competing models.

They coincide exactly under the idealized assumptions: the repo rate used is
the *general collateral* (risk-free/funding) rate with no specialness, it is
deterministic (or at least the same rate used for discounting and
compounding), and there is no counterparty credit spread. Repo specialness
(Q5) is precisely the friction that breaks the clean equivalence, and is
where the desk's edge, and the only "market risk" left after the hedge,
actually lives.
